In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.disk import DiskImage, DiskBooleanMask
from mtrain.utils import mkdir
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil
from mtrain.smallnet.unet.extract.draw import overlay_mask_on_img
from mtrain.utils import show
from fastai.data.core import DataLoaders, default_device
from collections import defaultdict
from torchvision import tv_tensors
from torchvision.transforms import v2
from PIL import Image
from fastai.vision.all import (
    vision_learner,
    mobilenet_v3_small,
    mobilenet_v3_large,
    accuracy,
    F1Score,
    CrossEntropyLossFlat,
    ProgressCallback,
)

In [ ]:
DS = Path("../../datasets/")
BASE = DS / "test-samples"
NEG_MASKING_V1 = BASE / "neg-masking" / "V1"
ROCKS = NEG_MASKING_V1 / "rocks"
SAMPLES_MAPILLARY = NEG_MASKING_V1 / "samples_mapillary"
ROCKS_TXT_FILE = ROCKS / "clip_query_res.txt"
ROCKS_DATA_DIR = mkdir(ROCKS / "data")
ROCKS_CVAT_DIR = mkdir(ROCKS / "cvat")
ROCKS_DS_DIR = ROCKS / "classification" / "crop_level"

SAMPLES_MAPILLARY.exists(), ROCKS.exists()
BASE_MODEL_DIR = Path("../../datasets/models")

# function defs

In [ ]:
def show_masks_unique(dirs, fname):
    uq = set()
    for d in tqdm(dirs):
        uq.add(tuple(np.unique(DiskBooleanMask.load(d / fname))))
    return uq


def fix_masks_unique(dirs):
    for d in tqdm(dirs):
        mask = DiskBooleanMask.load(d / "mask.png")
        mask = mask.astype(bool).astype(np.uint8)
        DiskBooleanMask.save(mask, d / "m1.png")


def make_m1_mask(dirs):
    for d in tqdm(dirs):
        shutil.move(d / "m1.png", d / "mask.png")


def verify_m1_and_masks(dirs):
    for d in tqdm(dirs):
        mask, m1 = d / "mask.png", d / "m1.png"
        mask, m1 = DiskBooleanMask.load(mask), DiskBooleanMask.load(m1)
        assert np.all(mask.astype(bool) == m1.astype(bool))
    print("all good")

In [ ]:
def _is_valid_dir(direc: Path):
    valid = (
        direc.is_dir()
        and (direc / "image.jpg").exists()
        and (direc / "mask.png").exists()
    )
    return valid


def get_ds_dirs(ds_root, labels):
    res = []
    for label in labels:
        cls_root = ds_root / label
        print(cls_root.name)
        if not cls_root.is_dir():
            continue
        for d in cls_root.glob("*"):
            if _is_valid_dir(d):
                res.append(d)
    random.shuffle(res)
    return res


def _label_func(d: Path):
    return Path(d).parent.name


def _validate_labels_in_dirs(dirs, label_by_idx):
    for d in dirs:
        label = _label_func(d)
        if label not in label_by_idx:
            raise Exception(
                f"Label={label} not found for directory={d}. label_by_index={label_by_idx}"
            )

MASK_DS_IMAGENET_MEAN = [0.485, 0.456, 0.406]
MASK_DS_IMAGENET_STD = [0.229, 0.224, 0.225]
MASK_DS_EVAL_TFMS = v2.Compose(
    [
        v2.Resize(130, antialias=True),
        v2.CenterCrop(130),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=MASK_DS_IMAGENET_MEAN, std=MASK_DS_IMAGENET_STD),
        v2.ToPureTensor(),
    ]
)

class MaskClassificationDataset(torch.utils.data.Dataset):
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

    def __init__(
        self,
        dirs: list[Path | str],
        norm_stats: tuple[list[float], list[float]],
        labels: list[
            str
        ],  # make sure these are always in the same order, new labels should be appended
        train: bool,
    ):
        self.mean = torch.tensor(norm_stats[0])
        self.std = torch.tensor(norm_stats[1])
        self.dirs = [d for d in map(Path, dirs) if _is_valid_dir(d)]
        self.labels = labels
        self.label_by_idx = {label: i for i, label in enumerate(self.labels)}
        train_tfms = v2.Compose(
            [
                v2.Resize(200, antialias=True),
                v2.RandomCrop(130),
                v2.RandomHorizontalFlip(p=0.5),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=MASK_DS_IMAGENET_MEAN, std=MASK_DS_IMAGENET_STD),
                v2.ToPureTensor(),
            ]
        )
        # eval_tfms = v2.Compose(
        #     [
        #         v2.Resize(130, antialias=True),
        #         v2.CenterCrop(130),
        #         v2.ToDtype(torch.float32, scale=True),
        #         v2.Normalize(mean=self.IMAGENET_MEAN, std=self.IMAGENET_STD),
        #         v2.ToPureTensor(),
        #     ]
        # )
        self.tfms = train_tfms if train else MASK_DS_EVAL_TFMS
        _validate_labels_in_dirs(self.dirs, self.label_by_idx)

    def __len__(self):
        return len(self.dirs)

    def __getitem__(self, index):
        d = self.dirs[index]

        # load individual as tensors
        img = tv_tensors.Image(Image.open(d / "image.jpg").convert("RGB"))
        mask = tv_tensors.Mask(Image.open(d / "mask.png").convert("L"))

        # tfms then combine
        t_img, t_mask = self.tfms([img, mask])
        combined = torch.cat([t_img, t_mask], dim=0)

        # label tensor
        label = _label_func(d)
        label_idx = self.label_by_idx[label]
        label_tensor = torch.Tensor([label_idx]).squeeze()

        return (combined, label_tensor)


class MaskInferenceDataset(torch.utils.data.Dataset):
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

    def __init__(self, img_mask_pairs):
        self.pairs = img_mask_pairs
        self.tfms = MASK_DS_EVAL_TFMS
        # self.tfms = v2.Compose([
        #     v2.Resize(130, antialias=True),
        #     v2.CenterCrop(130),
        #     v2.ToDtype(torch.float32, scale=True),
        #     v2.Normalize(mean=norm_stats[0], std=norm_stats[1]),
        #     v2.ToPureTensor(),
        # ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        img, mask = self.pairs[index]
        img = tv_tensors.Image(torch.from_numpy(img).permute(2, 0, 1))
        mask = tv_tensors.Mask(torch.from_numpy(mask).unsqueeze(0))
        t_img, t_mask = self.tfms([img, mask])
        return torch.cat([t_img, t_mask], dim=0)



In [ ]:
LABELS = ["trash", "other"]
AREA_THRES = 5

def is_area_greater(d, pixels_count_thres: int):
    n = np.array(Image.open(d / "mask.png").convert("L")).sum()
    return n > pixels_count_thres

def get_area(d):
    return np.array(Image.open(d / "mask.png").convert("L")).sum()

In [ ]:
def denormalize(tf_img):
    std = torch.Tensor(MaskClassificationDataset.IMAGENET_STD)
    mean = torch.Tensor(MaskClassificationDataset.IMAGENET_MEAN)
    tf_img = (tf_img * std + mean).clamp(0, 1)
    return tf_img

def get_image_from_input_tensor(t_img):
    return denormalize(t_img[:3].permute(1,2,0))


def show_single(d, data_point):
    orig_img, orig_mask = (
        plt.imread(d / "image.jpg"),
        plt.imread(d / "mask.png"),
    )
    tf_img, tf_mask = data_point[0][:3].permute(1, 2, 0), data_point[0][3]
    tf_img = denormalize(tf_img)
    tf_img, tf_mask = (tf_img.numpy() * 255).astype(np.uint8), tf_mask.numpy().astype(bool)
    show(
        [
            orig_img,
            orig_mask,
            overlay_mask_on_img(orig_img, orig_mask.astype(bool)),
            tf_img,
            tf_mask,
            overlay_mask_on_img(tf_img, tf_mask),
        ],
        (10, 10),
        ncols=3,
        axis="off",
    )


def get_label_stats(dirs):
    counts = defaultdict(lambda: 0)
    for d in dirs:
        counts[_label_func(d)] += 1
    return {
        "counts": dict(counts)
    }

In [ ]:
# load a learner
def load_our_learner(dls, model_arch, weights, pth_path=None):
    learn = vision_learner(
        dls,
        model_arch,
        n_in=4,
        metrics=[accuracy, F1Score(average="macro")],
        loss_func=CrossEntropyLossFlat(weight=weights),
        n_out=len(LABELS),
        normalize=False,
    )
    learn = learn.remove_cb(ProgressCallback)
    if pth_path is not None:
        pth_path = Path(pth_path)
        if pth_path.suffix == '.pth':
            print("WARN: the path you should pass should not have suffix .pth")
        learn = learn.load(pth_path)
    return learn

In [ ]:
def show_confusion_matrix(learn, dataloader):
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    preds, targs = learn.get_preds(dl=dataloader)
    pred_classes = preds.argmax(dim=1)

    cm = confusion_matrix(targs, pred_classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
    disp.plot()

def show_confusion_matrix_using_preds(learn, preds, targs):
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    pred_classes = preds.argmax(dim=1)

    cm = confusion_matrix(targs, pred_classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
    disp.plot()


def draw_mask_bboxes(ax, mask):
    mask_uint8 = mask.astype(np.uint8)
    n, labeled = cv2.connectedComponents(mask_uint8)
    for i in range(1, n):  # skip 0 (background)
        ys, xs = np.where(labeled == i)
        y0, y1 = ys.min(), ys.max()
        x0, x1 = xs.min(), xs.max()
        rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                              linewidth=1, edgecolor='red', facecolor='none')
        ax.add_patch(rect)

def show_images(dataset, idxs, preds, title, nrows=4):
    n = nrows * 4
    fig, axes = plt.subplots(nrows, 4, figsize=(30,30))
    fig.suptitle(title)
    for ax, idx in zip(axes.flat, idxs[:n]):
        img, label = dataset[idx.item()]
        t_img = denormalize(img[:3].permute(1,2,0))
        t_mask = img[3].numpy().astype(bool)
        ax.imshow(t_img)  # adjust if your img has 4 channels
        draw_mask_bboxes(ax, t_mask)
        score = preds[idx.item(), 0].item()  # prob of trash class
        ax.set_title(f"{idx.item()} | trash: {score:.2f}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()


# def get_preds_for_valid_ds(learn, valid_ds):
#     from torch.utils.data import DataLoader as TorchDataLoader
#     val_loader = TorchDataLoader(valid_ds, batch_size=4, shuffle=False)
#     learn.model.eval()
#     learn.model.to(default_device())
#     all_preds, all_targs = [], []
#     with torch.no_grad():
#         for x, y in val_loader:
#             x = x.to(default_device())
#             preds = learn.model(x).softmax(dim=1)
#             all_preds.append(preds.cpu())
#             all_targs.append(y.cpu())
#     all_preds = torch.cat(all_preds)
#     all_targs = torch.cat(all_targs)
#     decoded = all_preds.argmax(dim=1)
#     TRASH, OTHER = 0, 1  # adjust to match your label encoding
#     fp_idxs = ((all_targs == OTHER) & (decoded == TRASH)).nonzero().squeeze()
#     fn_idxs = ((all_targs == TRASH) & (decoded == OTHER)).nonzero().squeeze()

#     return all_preds, all_targs, decoded, fp_idxs, fn_idxs

def get_preds_for_valid_ds(learn, valid_ds):
    from torch.utils.data import DataLoader as TorchDataLoader
    import torch.nn.functional as F
    val_loader = TorchDataLoader(valid_ds, batch_size=4, shuffle=False)
    learn.model.eval()
    learn.model.to(default_device())
    all_preds, all_targs, all_losses = [], [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(default_device())
            logits = learn.model(x)
            preds = logits.softmax(dim=1)
            losses = F.cross_entropy(logits, y.to(default_device()), reduction='none')
            all_preds.append(preds.cpu())
            all_targs.append(y.cpu())
            all_losses.append(losses.cpu())
    all_preds = torch.cat(all_preds)
    all_targs = torch.cat(all_targs)
    all_losses = torch.cat(all_losses)
    decoded = all_preds.argmax(dim=1)
    TRASH, OTHER = 0, 1
    fp_idxs = ((all_targs == OTHER) & (decoded == TRASH)).nonzero().squeeze()
    fn_idxs = ((all_targs == TRASH) & (decoded == OTHER)).nonzero().squeeze()
    return all_preds, all_targs, decoded, fp_idxs, fn_idxs, all_losses

# Model training

## Dataset

In [ ]:
total_dirs = get_ds_dirs(ROCKS_DS_DIR, LABELS)
areas_and_dirs = [(get_area(d), d) for d in total_dirs]
dirs = [d for (a,d) in areas_and_dirs if a > AREA_THRES or a == 0]

print(f"total directories scanned: {len(total_dirs)}")
print(f"filtered directories: {len(dirs)}")


labels = [_label_func(d) for d in dirs]
train_dirs, valid_dirs = train_test_split(
    dirs, test_size=0.2, stratify=labels, random_state=42
)

In [ ]:
from fastai.vision.all import imagenet_stats

train_ds = MaskClassificationDataset(train_dirs, imagenet_stats, LABELS, train=True)
valid_ds = MaskClassificationDataset(valid_dirs, imagenet_stats, LABELS, train=False)

In [ ]:
dls = DataLoaders.from_dsets(train_ds, valid_ds, device=default_device())
LABELS, train_ds.label_by_idx, get_label_stats(dirs)

### Visualise single tensors

In [ ]:
# train test, see if transforms are working correctly
# you should see synced randomized cropping on multiple calling
idx = 9
show_single(train_dirs[idx], train_ds[idx])

In [ ]:
# valid test, see if transforms are working correctly
# no randomized cropping for flipping on multiple calls
idx = 11
show_single(valid_dirs[idx], valid_ds[idx])

# Model train

- Current status: very imbalanced, need more points for trash and unknown

In [ ]:
counts = torch.tensor([1000, 2000], dtype=torch.float32)
weights = 1 / counts
weights = weights / weights.sum()  # normalize
weights = weights.to(default_device())

learn = load_our_learner(dls, mobilenet_v3_large, weights)

learn = vision_learner(
    dls,
    mobilenet_v3_large,
    n_in=4,
    metrics=[accuracy, F1Score(average="macro")],
    loss_func=CrossEntropyLossFlat(weight=weights),
    n_out=len(LABELS),
    normalize=False,
)
learn = learn.remove_cb(ProgressCallback)

In [ ]:
learn.fine_tune(5)

In [ ]:
learn.fit_one_cycle(20, lr_max=slice(1e-4, 1e-3))

In [ ]:
model_dir = mkdir(DS / "models" / "trash_classification")
learn.save((model_dir / "mobilenet_large_mask_thres_6_25_epochs").resolve())

# MobileNet Small Visualize

In [ ]:
model_path = (BASE_MODEL_DIR / "trash_classification" / "mobilenet_small").resolve()
learn = load_our_learner(dls, mobilenet_v3_small, weights, model_path)
all_preds, _, fp_idxs, fn_idxs = get_preds_for_valid_ds(learn, valid_ds)

In [ ]:
show_images(valid_ds, fp_idxs, all_preds, "False Positives (predicted trash, actually other)")
show_images(valid_ds, fn_idxs, all_preds, "False Negatives (predicted other, actually trash)")

# Mobilenet large visualize

In [ ]:
model_path = (BASE_MODEL_DIR / "trash_classification" / "mobilenet_large").resolve()
learn = load_our_learner(dls, mobilenet_v3_large, None, model_path)

## Classification interpretation

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_4 import LossWidget

In [ ]:
widget = LossWidget(learn, valid_ds, 100, dry_run=False, descending=False)

In [ ]:
widget.ui()

In [ ]:
all_valid_preds, all_valid_targs, _, valid_fp_idxs, valid_fn_idxs, all_valid_losses = get_preds_for_valid_ds(learn, valid_ds)
all_train_preds, all_train_targs, _, train_fp_idxs, train_fn_idxs, all_train_losses = get_preds_for_valid_ds(learn, train_ds)

In [ ]:
train_losses_and_idxes = list(reversed(sorted((loss,i) for i, loss in enumerate(all_train_losses))))
valid_losses_and_idxes = list(reversed(sorted((loss,i) for i, loss in enumerate(all_valid_losses))))

In [ ]:
import itertools
bad_train_dirs = ((idx, train_dirs[idx]) for _, idx in train_losses_and_idxes)
img_and_masks = (
    (
        DiskImage.load(d / "image.jpg"), 
        DiskBooleanMask.load(d / "mask.png"), 
        (all_train_preds[idx], all_train_targs[idx])
    )
    for (idx,d) in bad_train_dirs
)
img_and_masks = (r for r in img_and_masks if r[1].sum() > 0)
with_overlaid = ((img, mask, overlay_mask_on_img(img, mask.astype(bool)), pred) for (img, mask, pred) in img_and_masks)

In [ ]:
print(next(with_overlaid)[-1])
show(next(with_overlaid)[:-1], (20,20), ncols=3)

In [ ]:
show_confusion_matrix_using_preds(learn, all_valid_preds, all_valid_targs)

In [ ]:
show_confusion_matrix_using_preds(learn, all_train_preds, all_train_targs)

In [ ]:
show_images(train_ds, train_fn_idxs, all_train_preds, "Train: False Negatives (predicted other, actually trash)")

In [ ]:
show_images(valid_ds, valid_fn_idxs, all_valid_preds, "Validation: False Negatives (predicted other, actually trash)")

In [ ]:
def mask_size_stats(pixel_counts):
    arr = np.array(pixel_counts)
    print(f"Count:    {len(arr)}")
    print(f"Mean:     {arr.mean():.1f}")
    print(f"Median:   {np.median(arr):.1f}")
    print(f"Std:      {arr.std():.1f}")
    print(f"Min:      {arr.min()}")
    print(f"Max:      {arr.max()}")
    print(f"25th pct: {np.percentile(arr, 25):.1f}")
    print(f"75th pct: {np.percentile(arr, 75):.1f}")
    print(f"95th pct: {np.percentile(arr, 95):.1f}")

    # fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # axes[0].hist(arr, bins=50, edgecolor='black')
    # axes[0].set_title('Distribution of Mask Pixel Counts')
    # axes[0].set_xlabel('Pixel Count')
    # axes[0].set_ylabel('Frequency')

    # axes[1].hist(arr, bins=50, edgecolor='black')
    # axes[1].set_yscale('log')
    # axes[1].set_title('Distribution (Log Scale)')
    # axes[1].set_xlabel('Pixel Count')
    # axes[1].set_ylabel('Frequency (log)')

    # for ax in axes:
    #     ax.axvline(np.median(arr), color='red', linestyle='--', label='Median')
    #     ax.axvline(arr.mean(), color='orange', linestyle='--', label='Mean')
    #     ax.legend()

    # plt.tight_layout()
    # plt.show()

In [ ]:
valid_fn_areas = [valid_ds[t][0][3].sum() for t in valid_fn_idxs]
valid_areas = [valid_ds[t][0][3].sum() for t in range(len(valid_ds))]
train_fn_areas = [train_ds[t][0][3].sum() for t in train_fn_idxs]
train_areas = [train_ds[t][0][3].sum() for t in range(len(train_ds))]

In [ ]:
mask_size_stats(train_areas)

In [ ]:
mask_size_stats(train_fn_areas)

In [ ]:
mask_size_stats(valid_areas)

In [ ]:
mask_size_stats(valid_fn_areas)

# Inference pipeline

In [ ]:
from mtrain.neg_mask.widget_2 import get_crops_for_image


def read_clip_file(path) -> list[tuple[str, Path]]:
    with open(path) as f:
        lines = f.readlines()
    imgs = [Path(line.split("\t")[1].strip()) for line in lines]
    dirs = [(path.stem, img.parent) for img in imgs]
    return dirs


paths = read_clip_file(NEG_MASKING_V1 / "trash" / "clip_delhi_litter.txt")

In [ ]:
idx = 31
plt.imshow(plt.imread(paths[idx][1] / "image.jpg"))

In [ ]:
d = paths[31][1]
img, mask = DiskImage.load(d / "image.jpg"), DiskBooleanMask.load(d / "m2.png")

In [ ]:
show([img, overlay_mask_on_img(img, mask.astype(bool))])

In [ ]:
def predict_trash(learn, img_mask_pairs, trash_pred_idx=0, threshold=0.25):
    from torch.utils.data import DataLoader as TorchDataLoader
    
    ds = MaskInferenceDataset(img_mask_pairs)
    dl = TorchDataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
    
    learn.model.eval()
    learn.model.to(default_device())
    
    all_probs = []
    with torch.no_grad():
        for x in dl:
            x = x.to(default_device())
            probs = learn.model(x).softmax(dim=1)
            all_probs.append(probs.cpu())
    
    all_probs = torch.cat(all_probs)
    trash_probs = all_probs[:, trash_pred_idx]
    predicted_trash = trash_probs >= threshold
    
    return predicted_trash, trash_probs

def predict_and_reconstruct_mask(learn, image: np.ndarray, mask: np.ndarray, trash_pred_idx=0, bbox_pad=20, crop_pad=220):
    bboxes, imgs, masks = [], [], []
    for bbox, crop_img, crop_mask in get_crops_for_image(image, mask, bbox_pad, crop_pad):
        bboxes.append(bbox)
        imgs.append(crop_img)
        masks.append(crop_mask)
    if not bboxes:
        return mask.copy().astype(np.float32)

    _, trash_probs = predict_trash(learn, list(zip(imgs, masks)), trash_pred_idx=trash_pred_idx, threshold=0)

    reconstructed = mask.copy().astype(np.float32)
    for bbox, prob in zip(bboxes, trash_probs):
        x, y, w, h = bbox.x, bbox.y, bbox.w, bbox.h
        region = reconstructed[y:y+h, x:x+w]
        region[region == 1] = prob.item()

    return reconstructed


def get_trash_mask(reconstructed: np.ndarray, threshold=0.25) -> np.ndarray:
    result = reconstructed.copy()
    result[(reconstructed > 0) & (reconstructed < threshold)] = 2
    result[reconstructed >= threshold] = 1
    return result

In [ ]:
new_mask = predict_and_reconstruct_mask(learn, img, mask)

In [ ]:
# generate the masks first
from fastai.vision.all import load_learner
learner100 = load_learner(
    "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-v3/log/export_iter_14.pkl"
)
learner50 = load_learner(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/iter_4_engulf_t009_more-skew-resnet18-50x50-v2/model.pkl"
)
SIZE = 100

In [ ]:
from mtrain.smallnet.unet.predict.strided import single
mask_100 = single.strided_predict_unet_only_mask(img, 100, learner100, [33, 66], 1)

In [ ]:
mask_50 = single.strided_predict_unet_only_mask(img, 50, learner50, [25], 1)

In [ ]:
O = overlay_mask_on_img
show([O(img, mask.astype(bool)), O(img, mask_100.astype(bool)), O(img, mask_50.astype(bool))], (30,30), ncols=3, axis="off")

In [ ]:
thres_25 = get_trash_mask(new_mask, 0.25)
thres_50 = get_trash_mask(new_mask, 0.5)
only_in_25 = (thres_25 == 1) & (thres_50 != 1)

show([
    thres_25, thres_50, only_in_25
], (20,20), 3, axis='off')

In [ ]:
show([
    overlay_mask_on_img(img, mask.astype(bool)), mask, 

    overlay_mask_on_img(img, get_trash_mask(new_mask) == 1),
    overlay_mask_on_img(img, get_trash_mask(new_mask, 0.5) == 1),
], (20,20), axis='off')